# **MiniMax-H3 (T2I T2V I2I I2V R2V)**
- Github repo: https://github.com/MiniMax-AI/MiniMax-H3

- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

- [Usage Guide](https://penioj.blogspot.com/2026/10/a-quick-guide-to-image-and-video.html)



In [ ]:
# @title 1. 💥 Prepare Environment & Download Models {"single-column":true}
import os
import sys
import shutil
import subprocess
import re
import time
import gc
from pathlib import Path
from IPython.display import clear_output, display, HTML, Image as IPImage

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Directory alignment & Repositories Setup
%cd /content
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI_MiniMax_H3_Turbo"):
    !git clone https://github.com/Isi-dev/ComfyUI-MiniMax-H3-Turbo ComfyUI_MiniMax_H3_Turbo
if not os.path.exists("ComfyUI_GGUF"):
    !git clone --branch leejet_27_09_2026 https://github.com/Isi-dev/ComfyUI_GGUF.git
if not os.path.exists("ComfyUI_VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite ComfyUI_VideoHelperSuite
if not os.path.exists("ComfyUI_KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes ComfyUI_KJNodes

%cd /content/ComfyUI
!pip install -r requirements.txt
!pip install -r custom_nodes/ComfyUI_VideoHelperSuite/requirements.txt
!pip install -r custom_nodes/ComfyUI_KJNodes/requirements.txt
!pip install gguf huggingface_hub hf_transfer sageattention triton imageio imageio-ffmpeg opencv-python color-matcher
!apt -y install -qq aria2 ffmpeg

clear_output()
print("✅ Repositories & core requirements installed.")

# @markdown ---
# @markdown ### **Authentication & Tokens**
hf_token = "" # @param {type:"string"}
civitai_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Core Model Selection**
img_2_video_model = "https://huggingface.co/unsloth/MiniMax-H3-GGUF/resolve/main/minimax_h3_fl2va_pruned-Q4_K.gguf" # @param {type:"string"}
ref_2_video_model = "https://huggingface.co/unsloth/MiniMax-H3-GGUF/resolve/main/minimax_h3_ref2va_pruned-Q4_K.gguf" # @param {type:"string"}
text_encoder_model = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}
video_vae_model = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors" # @param {type:"string"}
audio_vae_model = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### **Turbo LoRA**
download_turbo_lora = True # @param {type:"boolean"}

turbo_lora_file = "https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora/resolve/main/minimax_h3_turbo_v4_step600_ema.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### **Optional Custom LoRAs (HuggingFace or Civitai URLs / Repos)**
download_lora_1 = True # @param {type:"boolean"}
lora_1_url = "https://civitai.com/api/download/models/3260165?fileId=3143491" # @param {type:"string"}

download_lora_2 = False # @param {type:"boolean"}
lora_2_url = "" # @param {type:"string"}

download_lora_3 = False # @param {type:"boolean"}
lora_3_url = "" # @param {type:"string"}

download_lora_4 = False # @param {type:"boolean"}
lora_4_url = "" # @param {type:"string"}

download_lora_5 = False # @param {type:"boolean"}
lora_5_url = "" # @param {type:"string"}

# Set HF transfer environment
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download

def download_hf_file(repo_id, repo_relative_path, local_destination, token=None):
    if not repo_relative_path or repo_relative_path == "None":
        return None
    os.makedirs(local_destination, exist_ok=True)
    filename = os.path.basename(repo_relative_path)
    destination = os.path.join(local_destination, filename)
    if os.path.isfile(destination):
        print(f"⏭️ Already exists: {destination}")
        return filename
    print(f"⬇️ Downloading HF File: {repo_id}/{repo_relative_path}")
    try:
        downloaded_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_relative_path,
            token=token if token else None
        )
        shutil.copy2(downloaded_path, destination)
        try:
            os.remove(downloaded_path)
        except Exception:
            pass
        print(f"✅ Downloaded: {destination}")
        return filename
    except Exception as e:
        print(f"❌ Download failed for {repo_relative_path}: {e}")
        return None

def download_civitai_model(civitai_link, token, folder="models/loras"):
    os.makedirs(folder, exist_ok=True)
    try:
        model_id = civitai_link.split("/models/")[1].split("?")[0]
    except IndexError:
        raise ValueError("Invalid Civitai URL format. Expected: https://civitai.com/api/download/models/<ID>")
    civitai_url = f"https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor"
    if token:
        civitai_url += f"&token={token}"
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = f"civitai_lora_{timestamp}.safetensors"
    full_path = os.path.join(folder, filename)
    print(f"⬇️ Downloading Civitai model to {full_path}...")
    cmd = f"wget --max-redirect=10 --show-progress \"{civitai_url}\" -O \"{full_path}\""
    os.system(cmd)
    if os.path.exists(full_path) and os.path.getsize(full_path) > 0:
        print(f"✅ Civitai Download Complete: {filename}")
        return filename
    else:
        print(f"❌ Civitai Download failed.")
        return None

def download_generic_url(url, dest_dir, token_hf="", token_civitai=""):
    if not url or not url.strip():
        return None
    url = url.strip()
    if "civitai.com" in url.lower():
        return download_civitai_model(url, token_civitai, dest_dir)
    hf_match = re.search(r'huggingface\.co/([^/]+/[^/]+)/(?:resolve|blob)/[^/]+/(.+)', url)
    if hf_match:
        repo_id = hf_match.group(1)
        rel_path = hf_match.group(2)
        return download_hf_file(repo_id, rel_path, dest_dir, token_hf)
    # Fallback aria2c download
    filename = url.split('/')[-1].split('?')[0]
    os.makedirs(dest_dir, exist_ok=True)
    cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest_dir, '-o', filename, url]
    subprocess.run(cmd, check=False)
    return filename


# Process primary downloads
token_str = hf_token.strip()
civ_token_str = civitai_token.strip()

print("--- Downloading Base Models ---")

img2vid_model=download_generic_url(img_2_video_model, "models/diffusion_models", token_str, civ_token_str)
ref2vid_model=download_generic_url(ref_2_video_model, "models/diffusion_models", token_str, civ_token_str)
text_encoder=download_generic_url(text_encoder_model, "models/text_encoders", token_str, civ_token_str)
video_vae=download_generic_url(video_vae_model, "models/vae", token_str, civ_token_str)
audio_vae=download_generic_url(audio_vae_model, "models/vae", token_str, civ_token_str)

if download_turbo_lora and turbo_lora_file:
    turbo_lora_file=download_generic_url(turbo_lora_file, "models/loras", token_str, civ_token_str)

lora_files = []
lora_configs = [
    (download_lora_1, lora_1_url),
    (download_lora_2, lora_2_url),
    (download_lora_3, lora_3_url),
    (download_lora_4, lora_4_url),
    (download_lora_5, lora_5_url),
]

print("--- Downloading Custom LoRAs ---")
for idx, (enabled, url) in enumerate(lora_configs, 1):
    if enabled and url.strip():
        fname = download_generic_url(url, "models/loras", token_str, civ_token_str)
        if fname:
            lora_files.append(fname)
            print(f"LoRA {idx} registered: {fname}")


# Initialize ComfyUI Classes & Execution Pipeline {"single-column":true}
import torch
import numpy as np
import cv2
from PIL import Image
import imageio
from google.colab import files

sys.path.insert(0, '/content/ComfyUI')
from comfy import model_management
from nodes import (
    UNETLoader,
    CLIPLoader,
    VAELoader,
    VAEDecode,
    LoadImage,
    LoraLoaderModelOnly
)

# Custom Node Imports
try:
    from custom_nodes.websocket_image_save import SaveImageWebsocket
except ImportError:
    pass

from custom_nodes.ComfyUI_MiniMax_H3_Turbo import (
    MiniMaxH3TurboLoRA,
    MiniMaxH3TurboSampler
)

try:
    from comfy_extras.nodes_minimax_h3 import (
        MiniMaxH3ReferenceToVideo,
        MiniMaxH3ImageToVideo
    )
except ImportError:
    # Fallback mapping if nodes are exposed at root comfy_extras
    print("Could not find minimax node")

from comfy_extras.nodes_custom_sampler import BasicGuider, BasicScheduler, RandomNoise, KSamplerSelect, SamplerCustomAdvanced
from comfy_extras.nodes_video import CreateVideo, SaveVideo
from comfy_extras.nodes_audio import LoadAudio, VAEDecodeAudio

def upload_media_file(target_folder="/content/ComfyUI/input", prompt_text="Upload Reference File"):
    os.makedirs(target_folder, exist_ok=True)
    print(f"📤 {prompt_text}:")
    uploaded = files.upload()
    paths = []
    for filename in uploaded.keys():
        src_path = f"/content/ComfyUI/{filename}"
        dest_path = os.path.join(target_folder, filename)
        if os.path.exists(src_path):
            shutil.move(src_path, dest_path)
        paths.append(dest_path)
        print(f"Saved to: {dest_path}")
    return paths[0] if paths else None

def display_video(video_path):
    from base64 import b64encode
    if not os.path.exists(video_path):
        print(f"File not found: {video_path}")
        return
    video_data = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()
    display(HTML(f'''
    <video width=640 height=360 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))


clear_output()
print("✅ All selected models and dependencies are ready.")

In [ ]:
# @markdown # 2. 📤 Upload Media Inputs (Optional)
num_ref_images = 3 # @param {type:"slider", min:0, max:10, step:1}
num_ref_audios = 0 # @param {type:"slider", min:0, max:10, step:1}

image_paths = []
audio_paths = []

for i in range(num_ref_images):
    path = upload_media_file(prompt_text=f"Upload Reference Image {i+1}")
    if path:
        image_paths.append(path)

for i in range(num_ref_audios):
    path = upload_media_file(prompt_text=f"Upload Reference Audio {i+1}")
    if path:
        audio_paths.append(path)

print(f"Total tracked inputs: {len(image_paths)} Images, {len(audio_paths)} Audios.")

In [8]:
# @title 3. 💥 Generate Image/Video {"single-column":true}
import random
import os
import gc
import time
import torch
from IPython.display import clear_output, display, Image as IPImage

# @markdown ### Generation Mode & Prompting
mode = "Text-to-Video (T2V)" # @param ["Text-to-Image (T2I)", "Text-to-Video (T2V)", "Image-to-Image (I2I)", "Image-to-Video (I2V)", "Reference-to-Video (Ref2VA) "]
prompt = "A cinematic shot of a majestic dragon soaring over snow-capped mountains and saying with a low voice to a group of one young man and two women who all look unfazed as their clothes responds to the winds, \"I am the great dragon! Bow before me.\", photorealistic, 2k resolution, 24fps " # @param {type:"string"}

# @markdown ### Canvas & Video Options
width = 1024 # @param {type:"integer"}
height = 576 # @param {type:"integer"}
duration_seconds = 10 # @param {type:"number"}
ref_image_size = "match" # @param ["match", "max"]

# @markdown ### Sampling & Seed
seed = 0 # @param {type:"integer"}
sampler_name = "res_multistep" # @param ["uni_pc", "uni_pc_bh2", "ddim","euler", "euler_cfg_pp", "euler_ancestral", "euler_ancestral_cfg_pp", "heun", "heunpp2","dpm_2", "dpm_2_ancestral","lms", "dpm_fast", "dpm_adaptive", "dpmpp_2s_ancestral", "dpmpp_2s_ancestral_cfg_pp", "dpmpp_sde", "dpmpp_sde_gpu","dpmpp_2m", "dpmpp_2m_cfg_pp", "dpmpp_2m_sde", "dpmpp_2m_sde_gpu", "dpmpp_3m_sde", "dpmpp_3m_sde_gpu", "ddpm", "lcm","ipndm", "ipndm_v", "deis", "res_multistep", "res_multistep_cfg_pp", "res_multistep_ancestral", "res_multistep_ancestral_cfg_pp","gradient_estimation", "er_sde", "seeds_2", "seeds_3"]
scheduler = "beta" # @param ["simple","normal","karras","exponential","sgm_uniform","ddim_uniform","beta","linear_quadratic","kl_optimal"]

steps = 6 # @param {type:"integer", min:1, max:50}
cfg = 1.0 # @param {type:"number", min:0.0, max:20.0}
use_turbo_lora = True # @param {type:"boolean"}
turbo_strength = 1.0 # @param {type:"number", min:0.0, max:2.0}

# @markdown ### Custom LoRAs Strengths
lora_1_strength = 0 # @param {type:"number", min:-2.0, max:2.0}
lora_2_strength = 0 # @param {type:"number", min:-2.0, max:2.0}
lora_3_strength = 0 # @param {type:"number", min:-2.0, max:2.0}
lora_4_strength = 0 # @param {type:"number", min:-2.0, max:2.0}
lora_5_strength = 0 # @param {type:"number", min:-2.0, max:2.0}

if seed == 0:
    seed = random.randint(1, 100000000000000)
print(f"Active Seed: {seed}")

# MiniMax-H3 requires pixel dimensions to be strictly divisible by 32
width = max(32, round(width / 32) * 32)
height = max(32, round(height / 32) * 32)

print(f"mode: {mode}")

# Calculate frame length
if "Image (T2I)" in mode or "Image (I2I)" in mode:
    print("An image will be generated.")
    length = 1
else:
    raw_frames = max(5, int(round(duration_seconds * 24)))
    length = raw_frames + (5 * (raw_frames % 17)) % 17

print(f"Configured Resolution: {width}x{height} | Target Frames: {length}")

def generate_minimax_media():
    # Initialize variables to None so the finally block can safely clean them up
    model = clip = vae_video = vae_audio = None
    cond_out = latent_out = sampled_latent = None
    decoded_images = decoded_audio = created_video = None

    try:
        with torch.inference_mode():
            # Initialize Model Loaders
            unet_loader = UNETLoader()
            clip_loader = CLIPLoader()
            vae_loader = VAELoader()
            load_img_node = LoadImage()
            load_audio_node = LoadAudio()
            lora_loader = LoraLoaderModelOnly()

            # unet_file = os.path.basename(unet_model)
            is_ref2va = "Ref2VA" in mode or "Image (I2I)" in mode
            if is_ref2va:
                unet_file = os.path.basename(ref2vid_model)
                print(f"🔄 Ref2VA Mode Detected: Selected UNet -> {unet_file}")
            else:
                unet_file = os.path.basename(img2vid_model)
                print(f"🔄 Standard Mode Detected: Selected UNet -> {unet_file}")
            text_enc_file = os.path.basename(text_encoder)
            video_vae_file = os.path.basename(video_vae)
            audio_vae_file = os.path.basename(audio_vae)

            print("Loading Diffusion Model, Encoders & VAEs...")
            if "gguf" in unet_file.lower():
                from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
                unet_loader = UnetLoaderGGUF()
                model = unet_loader.load_unet(unet_file.split(" ")[0])[0]
            else:
                model = unet_loader.load_unet(unet_file, weight_dtype="default")[0]
            clip = clip_loader.load_clip(text_enc_file, type="minimax", device="default")[0]
            vae_video = vae_loader.load_vae(video_vae_file)[0]
            vae_audio = vae_loader.load_vae(audio_vae_file)[0]

            # Chain Custom LoRAs 1 to 5
            lora_strengths = [lora_1_strength, lora_2_strength, lora_3_strength, lora_4_strength, lora_5_strength]
            for idx, l_fname in enumerate(globals().get('lora_files', [])):
                if idx < 5 and l_fname:
                    st = lora_strengths[idx]
                    model = lora_loader.load_lora_model_only(model, l_fname, st)[0]

            # Apply MiniMax H3 Turbo LoRA & Sampler
            if use_turbo_lora and os.path.exists(f"models/loras/{os.path.basename(turbo_lora_file)}"):
                print("Applying MiniMax-H3 Turbo LoRA...")
                model = MiniMaxH3TurboLoRA().apply_lora(model, os.path.basename(turbo_lora_file), turbo_strength, False)[0]

            sampler = MiniMaxH3TurboSampler().get_sampler()[0]
            if use_turbo_lora is False:
                sampler = KSamplerSelect().get_sampler(sampler_name=sampler_name)[0]

            # Prepare Inputs & Reference Tensors
            import cv2

            safe_image_paths = globals().get('image_paths', [])
            safe_audio_paths = globals().get('audio_paths', [])

            ref_images_dict = {}
            ref_audios_dict = {}
            input_video_for_concat = None # Track the video for later

            # STRICT MODE LOCK: Only load images/videos if the mode explicitly requires it
            needs_visual_ref = any(m in mode for m in ["I2I", "I2V", "Ref2VA"])

            if safe_image_paths and needs_visual_ref:
                for idx, path in enumerate(safe_image_paths, start=1):
                    if os.path.exists(path):
                        path_to_load = path

                        # Check if the uploaded file is a video
                        if path.lower().endswith(('.mp4', '.mov', '.avi', '.webm', '.mkv')):
                            print(f"Video detected for input {idx}. Extracting the last frame...")
                            input_video_for_concat = path

                            cap = cv2.VideoCapture(path)
                            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

                            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_count - 1))
                            ret, frame = cap.read()

                            if not ret:
                                cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, frame_count - 2))
                                ret, frame = cap.read()

                            if ret:
                                last_frame_path = os.path.join(os.path.dirname(path), f"extracted_last_frame_{idx}.png")
                                cv2.imwrite(last_frame_path, frame)
                                path_to_load = last_frame_path
                                print(f"Last frame extracted and saved to {last_frame_path}")
                            cap.release()

                        print(f"Loading Reference Image {idx}: {path_to_load}")
                        tensor = load_img_node.load_image(os.path.basename(path_to_load))[0]
                        ref_images_dict[f"ref_image_{idx}"] = tensor
            elif not needs_visual_ref:
                print("Text-to-Video/Image Mode detected. Bypassing uploaded visual references.")

            if safe_audio_paths:
                for idx, path in enumerate(safe_audio_paths, start=1):
                    if os.path.exists(path):
                        print(f"Loading Reference Audio Track {idx}: {path}")
                        audio_res = load_audio_node.load(os.path.basename(path))

                        if type(audio_res).__name__ == 'NodeOutput':
                            tensor = list(audio_res)[0]
                        elif hasattr(audio_res, "__iter__"):
                            tensor = list(audio_res)[0]
                        else:
                            tensor = audio_res[0] if isinstance(audio_res, tuple) else audio_res

                        ref_audios_dict[f"ref_audio_{idx}"] = tensor

            # Reference-To-Video (Ref2VA) Conditioning Encoding

            if "Ref2VA" in mode or "Image (I2I)" in mode:
                ref2video = MiniMaxH3ReferenceToVideo()
                result = ref2video.execute(
                    clip=clip,
                    vae=vae_video,
                    audio_vae=vae_audio,
                    prompt=prompt,
                    width=width,
                    height=height,
                    length=length,
                    ref_image_size=ref_image_size,
                    ref_images=ref_images_dict if ref_images_dict else None,
                    ref_audios=ref_audios_dict if ref_audios_dict else None
                )
            else:
                img2video = MiniMaxH3ImageToVideo()

                first_frame_tensor = ref_images_dict.get("ref_image_1") if ref_images_dict else None
                last_frame_tensor = ref_images_dict.get("ref_image_2") if ref_images_dict else None

                result = img2video.execute(
                    clip=clip,
                    vae=vae_video,
                    prompt=prompt,
                    width=width,
                    height=height,
                    length=length,
                    first_frame=first_frame_tensor,
                    last_frame=last_frame_tensor
                )


            clip = None
            gc.collect()
            torch.cuda.empty_cache()

            if type(result).__name__ == 'NodeOutput':
                cond_out, latent_out = list(result)[:2]
            elif hasattr(result, "__iter__"):
                cond_out, latent_out = list(result)[:2]
            else:
                cond_out, latent_out = result[0], result[1]

            # Sampling Execution
            clear_output()
            print("Generating media...")
            guider = BasicGuider().get_guider(model, cond_out)[0]
            noise = RandomNoise().get_noise(seed)[0]

            from comfy_extras.nodes_custom_sampler import BasicScheduler
            sigmas = BasicScheduler().get_sigmas(model, scheduler, steps, 1.0)[0]

            sampled_latent = SamplerCustomAdvanced().sample(
                noise=noise,
                guider=guider,
                sampler=sampler,
                sigmas=sigmas,
                latent_image=latent_out
            )[0]


            model = None
            gc.collect()
            torch.cuda.empty_cache()

            # Joint Audio-Video Latent Decoding

            decoded_images = VAEDecode().decode(vae_video, sampled_latent)[0]
            decoded_audio = VAEDecodeAudio().decode(vae_audio, sampled_latent)[0]


            vae_video = vae_audio = None
            gc.collect()
            torch.cuda.empty_cache()

            # Render & Save Output
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_prefix = f"MiniMax_H3_{timestamp}"

            if length == 1:
                print("Saving Image Output...")
                from nodes import SaveImage
                save_node = SaveImage()
                save_node.save_images(images=decoded_images, filename_prefix=output_prefix)

                out_dir = "/content/ComfyUI/output"
                matching_files = [os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.startswith(output_prefix) and f.endswith((".png", ".jpg"))]

                if matching_files:

                    display(IPImage(filename=matching_files[0]))
                else:
                    print("❌ Image generation finished, but output file could not be located.")

            else:

                CreateVideoNode = CreateVideo()

                video_result = CreateVideoNode.execute(
                    images=decoded_images,
                    fps=24.0,
                    audio=decoded_audio,
                    bit_depth=8,
                    color_space="sRGB",
                    codec="none"
                )

                if type(video_result).__name__ == 'NodeOutput':
                    created_video = list(video_result)[0]
                elif hasattr(video_result, "__iter__"):
                    created_video = list(video_result)[0]
                else:
                    created_video = video_result[0] if isinstance(video_result, tuple) else video_result

                import folder_paths
                from comfy_api.latest import Types

                width_vid, height_vid = created_video.get_dimensions()
                full_output_folder, filename, counter, _, _ = folder_paths.get_save_image_path(
                    output_prefix,
                    folder_paths.get_output_directory(),
                    width_vid,
                    height_vid
                )

                ext = Types.VideoContainer.get_extension("auto")
                final_output_path = os.path.join(full_output_folder, f"{filename}_{counter:05}_.{ext}")

                created_video.save_to(
                    final_output_path,
                    format=Types.VideoContainer("auto"),
                    codec="auto",
                    metadata=None
                )

                if os.path.exists(final_output_path):
                    # Intercept and concatenate if an input video was tracked AND we are actively using it
                    if input_video_for_concat and os.path.exists(input_video_for_concat) and needs_visual_ref:

                        joined_output_path = final_output_path.replace(".mp4", "_joined.mp4")

                        # Reverted to the simpler scaling concat that safely pads dimension mismatches without strictly cropping/dropping
                        import subprocess

                        # Detect if the original input video has an audio stream
                        probe_cmd = f'ffprobe -i "{input_video_for_concat}" -show_streams -select_streams a -loglevel error'
                        has_audio = subprocess.check_output(probe_cmd, shell=True).decode('utf-8').strip() != ""

                        # If no audio exists, create a temporary copy with an integrated silent track
                        if not has_audio:
                            silent_video_path = "temp_with_silence.mp4"
                            silence_cmd = f'ffmpeg -y -i "{input_video_for_concat}" -f lavfi -i anullsrc=channel_layout=stereo:sample_rate=48000 -c:v copy -c:a aac -shortest "{silent_video_path}" -loglevel error'
                            subprocess.run(silence_cmd, shell=True)
                            input_video_for_concat = silent_video_path  # Point to the newly silenced file for concatenation

                        # Safely execute the concatenation (Input 0 is now guaranteed to have [0:a])
                        ffmpeg_cmd = (
                            f'ffmpeg -y -i "{input_video_for_concat}" -i "{final_output_path}" '
                            f'-filter_complex "[0:v]scale={width}:{height}:force_original_aspect_ratio=decrease,pad={width}:{height}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps=24[v0]; '
                            f'[1:v]scale={width}:{height}:force_original_aspect_ratio=decrease,pad={width}:{height}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps=24[v1]; '
                            f'[v0][0:a][v1][1:a]concat=n=2:v=1:a=1[v][a]" '
                            f'-map "[v]" -map "[a]" -c:v libx264 -c:a aac -pix_fmt yuv420p "{joined_output_path}" -loglevel error'
                        )
                        os.system(ffmpeg_cmd)

                        if os.path.exists(joined_output_path):

                            final_output_path = joined_output_path

                    display_video(final_output_path)
                else:
                    print("❌ Video generation finished, but output file could not be located.")

    finally:
        model = clip = vae_video = vae_audio = cond_out = latent_out = sampled_latent = decoded_images = decoded_audio = created_video = None
        gc.collect()
        torch.cuda.empty_cache()


# Execute the pipeline
generate_minimax_media()

Generating media...


  0%|          | 0/6 [00:00<?, ?it/s]